# M1 — EDA, train, SHAP (beginner notebook)

**What this is:** run each cell top → bottom with `Shift + Enter`.

## Fix ModuleNotFoundError (important)

Those errors mean the notebook is using the **wrong Python** (system Python), not `ml/.venv`.

1. Click the **kernel name** at the top-right of this notebook.
2. Choose **Python (imfs-m1)**  
   or browse to: `ml\.venv\Scripts\python.exe`
3. **Restart** the kernel, then run cells from the top again.

Do **not** use a plain `Python 3.x` from `C:\Python314\...` — that one has no pandas/xgboost.

**Before first run** (terminal, venv active):
```powershell
cd ml
.\.venv\Scripts\Activate.ps1
python m1_credit_risk/scripts/generate_synthetic.py -n 3000
python m1_credit_risk/scripts/merge_par_synthetic.py
```

## 1) Imports and paths

In [ ]:
# Run this FIRST — installs missing packages into THIS notebook's kernel
import sys
print("Using Python:", sys.executable)

%pip install -q matplotlib pandas numpy scikit-learn xgboost shap joblib openpyxl

import matplotlib
import pandas
import sklearn
import xgboost
import shap
import joblib
print("Packages OK")
print("matplotlib", matplotlib.__version__)

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

# Notebook lives in ml/notebooks/m1 → repo ml/ is parents[2]
ML_ROOT = Path.cwd()
if ML_ROOT.name == "m1":
    ML_ROOT = ML_ROOT.parents[1]
elif (ML_ROOT / "m1_credit_risk").exists():
    pass
else:
    # fallback if started from repo root
    cand = ML_ROOT / "ml"
    ML_ROOT = cand if cand.exists() else ML_ROOT

M1 = ML_ROOT / "m1_credit_risk"
sys.path.insert(0, str(M1))

from features.schema import (
    BOOL_FEATURES,
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    TARGET,
    prepare_xy,
    risk_band,
)

DATA = M1 / "data" / "processed" / "train_par_synth_v1.csv"
ART = M1 / "artifacts"
ART.mkdir(parents=True, exist_ok=True)
print("ML_ROOT:", ML_ROOT)
print("DATA exists:", DATA.exists(), DATA)

## 2) Load merged PAR + synthetic data

In [ ]:
assert DATA.exists(), "Run merge_par_synthetic.py first (see markdown at top)."
df = pd.read_csv(DATA)
print("Rows:", len(df))
print(df["data_source"].value_counts())
df[[TARGET, "arrears_ratio", "paid_pct", "contract_amount"]].describe()

## 3) Quick EDA (charts for your report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df[TARGET].hist(bins=30, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Risk % distribution")
axes[0].set_xlabel("risk_pct")

df.boxplot(column=TARGET, by="data_source", ax=axes[1])
axes[1].set_title("Risk % by data source")
axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

print("Gender mix:\n", df["gender"].value_counts())
print("Area mix:\n", df["area_type"].value_counts())

## 4) Train XGBoost on risk_pct

In [ ]:
x, y = prepare_xy(df)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

pre = ColumnTransformer(
    [
        ("num", "passthrough", NUMERIC_FEATURES + BOOL_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)
model = XGBRegressor(
    n_estimators=120,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=4,
)
pipe = Pipeline([("pre", pre), ("model", model)])
pipe.fit(x_train, y_train)
pred = pipe.predict(x_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f"Test MAE: {mae:.3f}")
print(f"Test R2 : {r2:.3f}")

joblib.dump(pipe, ART / "risk_xgb.joblib")
print("Saved model ->", ART / "risk_xgb.joblib")

## 5) Predicted vs actual (scatter)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=0.25, s=12)
lims = [0, 100]
plt.plot(lims, lims, "r--", label="perfect")
plt.xlabel("Actual risk_pct")
plt.ylabel("Predicted risk_pct")
plt.title("Test set fit")
plt.legend()
plt.tight_layout()
plt.show()

## 6) Simple explanation for one loan (top factors)

Full SHAP can be slow; this uses XGBoost gain-style contribution via a TreeExplainer on one row.

In [ ]:
import shap

row_i = 0
row = x_test.iloc[[row_i]]
pct = float(pipe.predict(row)[0])
print("Predicted risk_pct:", round(pct, 2), "band:", risk_band(pct))

Xt = pipe.named_steps["pre"].transform(row)
explainer = shap.TreeExplainer(pipe.named_steps["model"])
sv = explainer.shap_values(Xt)
values = np.asarray(sv).reshape(-1)
names = pipe.named_steps["pre"].get_feature_names_out()
top = sorted(zip(names, values), key=lambda t: abs(t[1]), reverse=True)[:8]

for name, val in top:
    direction = "increases" if val > 0 else "decreases"
    print(f"  {name:40s}  shap={val:+.3f}  ({direction} risk)")

## Done

- Model file: `m1_credit_risk/artifacts/risk_xgb.joblib` (local only)
- For API demo: `python m1_credit_risk/serve/demo_predict.py`
- Public Kiva data is optional later (see `docs/SHARED_DATASETS.md`) — not required to finish this notebook.